In [1]:
# Types of conditional Routing 
# 1. Binary Condition(IF/ELSE)

In [1]:
from langgraph.graph import StateGraph
from typing import TypedDict

# Proper state schema  
class State(TypedDict): 
    number:int 
    result:str

    # Nodes 
def check_even(state): 
    return state

def even_node(state): 
        return{**state,'result':f'{state['number']} is EVEN'}
def odd_node(state): 
    return{**state,'result':f'{state['number']} is ODD'}
# Graph 
builder=StateGraph(State)

builder.add_node("check",check_even)
builder.add_node("even",even_node)
builder.add_node("odd",odd_node)

# Routing

def route(state):
    return 'even' if state['number'] % 2 ==0 else 'odd'
builder.add_conditional_edges(
    'check', 
    route,
    {
        'even':'even',
        'odd': 'odd'
    }
)
builder.set_entry_point('check')
graph=builder.compile() 

# Run 
print(graph.invoke({'number':7}))

{'number': 7, 'result': '7 is ODD'}


In [3]:
# 2. Multi-Condition Routing 

In [2]:
from langgraph.graph import StateGraph
from typing import TypedDict

# Proper state schema  
class State(TypedDict): 
    score:int 
    result:str

# Nodes 
def check_score(state):
   
    return state

def excellent_node(state):
    
    return {**state, "result": f"Score {state['score']} → EXCELLENT"}

def average_node(state):
    return {**state, "result": f"Score {state['score']} → AVERAGE"}

def fail_node(state):
    return {**state, "result": f"Score {state['score']} → FAIL"}

# Routing Logic 
def route_score(state): 
    score=state['score']
    if score >= 90: 
        return 'excellent' 
    elif score >= 50: 
        return  'average' 
    else : 
        return 'fail' 

# Build Graph 
builder=StateGraph(State)

builder.add_node('check_score',check_score)
builder.add_node('excellent_node',excellent_node) 
builder.add_node('average_node',average_node) 
builder.add_node('fail_node',fail_node)

# Condional Edges 

builder.add_conditional_edges(
    'check_score',
    route_score, 
    {
        'excellent': 'excellent_node',
        'average': 'average_node', 
        'fail': 'fail_node'
    }
)

# Entry Point 
builder.set_entry_point('check_score')

# Oprional but Recommended finish nodes 
builder.set_finish_point('excellent_node')
builder.set_finish_point('average_node') 
builder.set_finish_point('fail_node')

# compile 
graph=builder.compile() 

# Run Examples 
print(graph.invoke({'score':95}))
print(graph.invoke({'score':70})) 
print(graph.invoke({'score':30}))

{'score': 95, 'result': 'Score 95 → EXCELLENT'}
{'score': 70, 'result': 'Score 70 → AVERAGE'}
{'score': 30, 'result': 'Score 30 → FAIL'}


# condition loop (dynamic ittration )

In [6]:
from langgraph.graph import StateGraph
from typing import TypedDict

class State(TypedDict):
    attempt: int
    success: bool
    result: str

# create a node

In [7]:
def try_node(state):
    attempt = State.get("attempt",0) + 1

    success = attempt >= 3

    print(f"Attempt{attempt} -> {'Success' if success else 'Fail'}")

    return {
        **State,
        "attempt":attempt,
        "success":success
    }

# routing logic

In [9]:
def check_success(state):
    return "end" if state["success"] else "retry"


# Build Graph

In [11]:
builder = StateGraph(State)
builder.add_node("try_node",try_node)

# conditional loop

In [ ]:
builder.add_conditional_edges(
    "try_node",
    check_success,
    {
        "end":"__end__", # stop execution
        "retry":"try_node" # loop again
    }
)

In [21]:
# entry point 
builder.set_entry_point("try_node")

# compile
graph = builder.compile()

result = graph.invoke({
    "attempt":0,
    "succes":False,
    "result":" "
})


print("final Output")
print(result)

ValueError: At 'try_node' node, 'check_success' branch found unknown target '__end_'